# LC 901 — Online Stock Span
**Day-56 | Monotonic Stack**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A monotonic decreasing stack lets you
collapse all prior days whose price was &le; today's price into a
single span count — O(1) amortised per call because each price is
pushed and popped at most once.
</div>

## Official Problem Statement

Design an algorithm to collect daily stock prices and return the
**stock span** for the current day.

The stock span for the current day is the maximum number of
consecutive days (including today) for which the stock price was
less than or equal to today's price.

Implement the `StockSpanner` class:
- `StockSpanner()` — initialises the object.
- `int next(int price)` — returns the span of the stock price
  for today.

**Constraints:**
- `1 <= price <= 10^5`
- At most `10^4` calls to `next`.

## What This Is Actually Asking

Each day you receive a stock price and must instantly answer:
"how many consecutive days ending today had a price &le; today?"

A naive scan back through history is O(n) per call and too slow.

The trick is to store *compressed* history: only days that were
strictly higher than anything seen since, plus how many days they
already represent.

When today's price is &ge; a stored day, we absorb its span and
discard it — we'll never need it again.

## Walk Through an Example by Hand

Prices arrive: 100, 80, 60, 70, 60, 75, 85

```
Day 1: price=100  stack=[]           span=1  push (100,1)
Day 2: price=80   top=(100,1) 100>80 span=1  push (80,1)
Day 3: price=60   top=(80,1)  80>60  span=1  push (60,1)
Day 4: price=70   top=(60,1)  60<=70 pop, span+=1 -> span=2
                  top=(80,1)  80>70  stop    push (70,2)
Day 5: price=60   top=(70,2)  70>60  span=1  push (60,1)
Day 6: price=75   top=(60,1)  60<=75 pop, span+=1 -> span=2
                  top=(70,2)  70<=75 pop, span+=2 -> span=4
                  top=(80,1)  80>75  stop    push (75,4)
Day 7: price=85   top=(75,4)  75<=85 pop, span+=4 -> span=5
                  top=(80,1)  80<=85 pop, span+=1 -> span=6
                  top=(100,1) 100>85 stop    push (85,6)
```
Answers: 1, 1, 1, 2, 1, 4, 6

## The Picture

```
Price timeline (bars = height):

100 |X
 80 |  X
 75 |            X<-- span 4 (days 3-6)
 70 |      X
 60 |        X X
 85 |               X<-- span 6 (days 1-7, only 100 blocks)

     D1 D2 D3 D4 D5 D6 D7

Stack stores only the "skyline" — bars taller than everything
to their right.  Each bar carries a pre-computed span so when
it's popped we absorb its history in O(1).

  Stack (price, span)          Action
  ─────────────────────────────────────
  [(100,1)]                    after D1
  [(100,1),(80,1)]             after D2
  [(100,1),(80,1),(60,1)]      after D3
  [(100,1),(80,1),(70,2)]      after D4  -- absorbed (60,1)
  [(100,1),(80,1),(70,2),(60,1)] after D5
  [(100,1),(80,1),(75,4)]      after D6  -- absorbed (60,1),(70,2)
  [(100,1),(85,6)]             after D7  -- absorbed (75,4),(80,1)
```

## When To Use This Pattern

- When you need the **nearest greater element** to the left/right,
  think monotonic stack.
- When each element is pushed and popped exactly once, think
  O(n) amortised via monotonic stack.
- When history can be **compressed** into a single stored value
  (like a span), think stack of (value, aggregated-count).
- When a design problem requires O(1) amortised queries on a
  stream, think monotonic stack over a sorted structure.
- When you see "consecutive days", "temperature span", "histogram",
  think monotonic stack.

## The Approach

Maintain a stack of `(price, span)` tuples in monotonically
decreasing order of price.

For each new price, start with `span = 1`. Pop every stack entry
whose price is less than or equal to today's price, adding its
span to ours — those days are now fully covered by today.

Push `(price, span)` onto the stack and return `span`.

Total work across all calls is O(n) because every entry is pushed
once and popped at most once — O(1) amortised per call.

In [1]:
# Imports
from typing import List

In [2]:
# --------------- Test Harness (class-based) ---------------

def run_tests(SpannerClass):
    """Replay next() calls and compare against expected spans."""
    cases = [
        {
            "prices":   [100, 80, 60, 70, 60, 75, 85],
            "expected": [1,   1,  1,  2,  1,  4,  6],
            "label":    "LC example 1",
        },
        {
            "prices":   [10, 20, 30, 40],
            "expected": [1,  2,  3,  4],
            "label":    "strictly increasing",
        },
        {
            "prices":   [40, 30, 20, 10],
            "expected": [1,  1,  1,  1],
            "label":    "strictly decreasing",
        },
        {
            "prices":   [5, 5, 5, 5],
            "expected": [1, 2, 3, 4],
            "label":    "all equal",
        },
    ]

    passed = 0
    for c in cases:
        spanner = SpannerClass()
        results = [spanner.next(p) for p in c["prices"]]
        ok = results == c["expected"]
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(f"  {status} [{c['label']}]")
            print(f"    got:      {results}")
            print(f"    expected: {c['expected']}")
        print(f"  {status} [{c['label']}]")

    total = len(cases)
    print(f"\n{'='*40}")
    print(f"Results: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")
    else:
        print(f"{total - passed} test(s) FAILED.")

In [6]:
class StockSpanner:
    """
    LC 901 — Online Stock Span

    Strategy:
        Monotonic decreasing stack of (price, span).
        next(price):
          span = 1
          while stack and stack[-1][0] <= price:
              span += stack.pop()[1]
          stack.append((price, span))
          return span

    Time : O(1) amortised per call (each element pushed/popped once)
    Space: O(n) — at most n entries in the stack
    """

    def __init__(self):
        # TODO: initialise the stack
        self.stack = []            #insert price and span
        pass

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack[-1][1]
            self.stack.pop()
        self.stack.append([price, span])
        return span
# --- Debug prints (expected in comments) ---
s = StockSpanner()
print(s.next(100))  # 1
print(s.next(80))   # 1
print(s.next(60))   # 1
print(s.next(70))   # 2
print(s.next(60))   # 1
print(s.next(75))   # 4
print(s.next(85))   # 6

print("---")
s2 = StockSpanner()
print(s2.next(100))  # 1
print(s2.next(100))  # 2  (equal counts!)
print(s2.next(100))  # 3

run_tests(StockSpanner)


1
1
1
2
1
4
6
---
1
2
3
  PASSED [LC example 1]
  PASSED [strictly increasing]
  PASSED [strictly decreasing]
  PASSED [all equal]

Results: 4/4 passed
All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# run_tests(StockSpanner)

## Complexity

| Approach | Time (per call) | Space |
|---|---|---|
| Brute force — scan back | O(n) | O(n) |
| Monotonic stack (optimal) | O(1) amortised | O(n) |

Each price is pushed onto the stack once and popped at most once,
so across n calls the total work is O(n) — O(1) amortised.

Space is O(n) in the worst case (strictly decreasing prices mean
nothing is ever popped).

## Real World Connection

At Citi, risk desks compute rolling "days since last breach" for
VaR thresholds in real time — exactly the stock span problem on a
stream of risk metrics.

AWS CloudWatch Metrics uses a similar window-collapse technique
when computing "consecutive periods above alarm threshold" without
re-scanning the entire time series.

In data engineering, monotonic stacks appear in streaming
aggregations where you need running "days since last high" or
"streak length" on unbounded event logs.

The amortised O(1) guarantee makes this pattern critical for
high-frequency trading systems where microsecond latency matters
and O(n) rescans per tick are unacceptable.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra